# Notebook 05 - Machine Learning Modeling
**Proyek:** Pulsevera - Predict, Prevent, Prevail  
**Role:** AI Engineer (Fathan & Shafira)  
**Tujuan:** Training tiga model klasifikasi baseline (Logistic Regression, Random Forest, Decision Tree), menangani class imbalance (~5.6% kelas positif), melakukan hyperparameter tuning, dan menyimpan model terbaik untuk inferensi di FastAPI.

**Output utama:**
- `ml-api/models/pulsevera_ml_model.pkl` — model ML terbaik
- `ml-api/models/feature_metadata.json` — `FEATURE_ORDER` + statistik fitur
- `ml-api/models/ml_results.json` — ringkasan metrik tiap model

## 1. Setup & Import

In [ ]:
import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore')

BASE_DIR = Path('..')
FINAL_DIR = BASE_DIR / 'data' / 'final'
MODELS_DIR = BASE_DIR / 'ml-api' / 'models'
FIGURES_DIR = BASE_DIR / 'notebooks' / 'figures'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Setup OK.')

## 2. Load Data (output dari Data Scientist)

In [ ]:
X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_test = pd.read_csv(FINAL_DIR / 'X_test.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').squeeze()
y_test = pd.read_csv(FINAL_DIR / 'y_test.csv').squeeze()

FEATURE_ORDER = X_train.columns.tolist()

print(f'X_train: {X_train.shape}   y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}    y_test : {y_test.shape}')
print(f'Jumlah fitur : {len(FEATURE_ORDER)}')
print(f'Class balance train: {y_train.value_counts(normalize=True).round(4).to_dict()}')
print(f'Class balance test : {y_test.value_counts(normalize=True).round(4).to_dict()}')

## 3. Tangani Class Imbalance dengan SMOTE

Hanya ~5.64% kasus positif. Tanpa penanganan, model akan bias ke kelas negatif. Strategi:
- **SMOTE** (`sampling_strategy=0.3`) — naikkan rasio positif ke 30% pada training set saja.
- **Class weighting** sebagai backup pada tiap model (`class_weight='balanced'`).
- **JANGAN** terapkan SMOTE ke `X_test` — biarkan distribusi alami untuk evaluasi jujur.

In [ ]:
print('Distribusi sebelum SMOTE:')
print(y_train.value_counts().to_dict())

smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.3)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('\nDistribusi setelah SMOTE:')
print(pd.Series(y_train_res).value_counts().to_dict())
print(f'Shape resampled X_train: {X_train_res.shape}')

## 4. Scaling (penting untuk Logistic Regression)

StandardScaler hanya untuk Logistic Regression. Tree-based models (RF, DT) tidak butuh scaling.

In [ ]:
scaler = StandardScaler()
X_train_res_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
print(f'Scaler disimpan ke {MODELS_DIR / "scaler.pkl"}')

## 5. Baseline Models (3 algoritma wajib)

In [ ]:
def evaluate(name, y_true, y_pred, y_proba):
    """Hitung semua metrik penting dalam satu pemanggilan."""
    return {
        'model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_pos': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall_pos': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_pos': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba),
    }

baseline_results = []

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
t0 = time.time()
lr.fit(X_train_res_scaled, y_train_res)
elapsed = time.time() - t0

y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]
metrics_lr = evaluate('Logistic Regression', y_test, y_pred_lr, y_proba_lr)
metrics_lr['train_seconds'] = round(elapsed, 2)
baseline_results.append(metrics_lr)

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, digits=4))
print(f'ROC-AUC: {metrics_lr["roc_auc"]:.4f}   (training {elapsed:.1f}s)')

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=20, min_samples_split=5,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
t0 = time.time()
rf.fit(X_train_res, y_train_res)
elapsed = time.time() - t0

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
metrics_rf = evaluate('Random Forest', y_test, y_pred_rf, y_proba_rf)
metrics_rf['train_seconds'] = round(elapsed, 2)
baseline_results.append(metrics_rf)

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, digits=4))
print(f'ROC-AUC: {metrics_rf["roc_auc"]:.4f}   (training {elapsed:.1f}s)')

In [ ]:
dt = DecisionTreeClassifier(max_depth=12, class_weight='balanced', random_state=RANDOM_STATE)
t0 = time.time()
dt.fit(X_train_res, y_train_res)
elapsed = time.time() - t0

y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]
metrics_dt = evaluate('Decision Tree', y_test, y_pred_dt, y_proba_dt)
metrics_dt['train_seconds'] = round(elapsed, 2)
baseline_results.append(metrics_dt)

print('=== Decision Tree ===')
print(classification_report(y_test, y_pred_dt, digits=4))
print(f'ROC-AUC: {metrics_dt["roc_auc"]:.4f}   (training {elapsed:.1f}s)')

## 6. Ringkasan Performa Baseline

In [ ]:
df_baseline = pd.DataFrame(baseline_results)
display_cols = ['model', 'accuracy', 'precision_pos', 'recall_pos', 'f1_pos', 'roc_auc', 'train_seconds']
print('=== HASIL BASELINE ===')
print(df_baseline[display_cols].round(4).to_string(index=False))

print('\nTarget Coding Camp:')
print('  Accuracy   >= 0.85')
print('  Recall (1) >= 0.70  (prioritas — jangan lewatkan kasus risiko tinggi)')
print('  F1 (1)     >= 0.65')
print('  ROC-AUC    >= 0.80')

## 7. Hyperparameter Tuning *(opsional — set `DO_TUNE = False` untuk skip)*

Gunakan `RandomizedSearchCV` dengan scoring `f1` (BUKAN accuracy karena class imbalance).

> **Tip:** Untuk iterasi cepat, set `DO_TUNE = False` di cell di bawah. Notebook tetap jalan
> sampai akhir tanpa menyentuh tuning. Aktifkan kembali bila ingin model RF yang lebih optimal.

In [ ]:
# =====================================================================
# Set DO_TUNE = False bila ingin hasil cepat (skip RandomizedSearchCV)
# =====================================================================
DO_TUNE = False

# Container untuk model & metrik hasil tuning (kosong kalau DO_TUNE=False)
extra_models = {}
extra_results = []

if DO_TUNE:
    param_grid_rf = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'class_weight': ['balanced', 'balanced_subsample'],
    }

    rs_rf = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        param_distributions=param_grid_rf,
        n_iter=15, cv=3, scoring='f1', random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
    )
    t0 = time.time()
    rs_rf.fit(X_train_res, y_train_res)
    elapsed = time.time() - t0

    print(f'\nBest CV F1   : {rs_rf.best_score_:.4f}')
    print(f'Best params  : {rs_rf.best_params_}')
    print(f'Tuning time  : {elapsed/60:.1f} menit')

    rf_tuned = rs_rf.best_estimator_
    y_pred_rf_t = rf_tuned.predict(X_test)
    y_proba_rf_t = rf_tuned.predict_proba(X_test)[:, 1]
    metrics_rf_t = evaluate('Random Forest (tuned)', y_test, y_pred_rf_t, y_proba_rf_t)
    metrics_rf_t['best_params'] = rs_rf.best_params_

    extra_models['Random Forest (tuned)'] = (rf_tuned, metrics_rf_t, False)
    extra_results.append(metrics_rf_t)

    print('\n=== Random Forest (tuned) ===')
    print(classification_report(y_test, y_pred_rf_t, digits=4))
    print(f'ROC-AUC: {metrics_rf_t["roc_auc"]:.4f}')
else:
    print('SKIP hyperparameter tuning (DO_TUNE=False).')
    print('Set DO_TUNE = True untuk eksekusi RandomizedSearchCV (~5–15 menit).')

In [ ]:
# (cell ini sengaja dikosongkan — logic tuning sudah digabung ke cell di atas)

## 8. Pilih Model Terbaik & Simpan untuk FastAPI

Strategi pemilihan: prioritaskan **F1 kelas positif**, dengan recall minimal 70%. Bila tidak tercapai, fallback ke threshold tuning.

In [ ]:
all_models = {
    'Logistic Regression': (lr, metrics_lr, True),
    'Random Forest': (rf, metrics_rf, False),
    'Decision Tree': (dt, metrics_dt, False),
    **extra_models,   # akan menambahkan 'Random Forest (tuned)' bila DO_TUNE=True
}

best_name = max(all_models, key=lambda k: all_models[k][1]['f1_pos'])
best_model, best_metrics, best_needs_scaling = all_models[best_name]

print(f'Model terpilih: {best_name}')
print(f'  F1 (kelas 1)   : {best_metrics["f1_pos"]:.4f}')
print(f'  Recall (kelas 1): {best_metrics["recall_pos"]:.4f}')
print(f'  ROC-AUC        : {best_metrics["roc_auc"]:.4f}')
print(f'  Scaling needed : {best_needs_scaling}')

In [ ]:
model_path = MODELS_DIR / 'pulsevera_ml_model.pkl'
joblib.dump(best_model, model_path)
print(f'Model disimpan: {model_path} ({model_path.stat().st_size / 1024**2:.1f} MB)')

feature_metadata = {
    'feature_order': FEATURE_ORDER,
    'n_features': len(FEATURE_ORDER),
    'best_model_name': best_name,
    'needs_scaling': best_needs_scaling,
    'feature_means': X_train.mean().round(4).to_dict(),
    'feature_medians': X_train.median().round(4).to_dict(),
}
with open(MODELS_DIR / 'feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)
print(f'Feature metadata disimpan: {MODELS_DIR / "feature_metadata.json"}')

all_results = baseline_results + extra_results   # extra_results kosong bila DO_TUNE=False
results_summary = {m['model']: {k: v for k, v in m.items() if k != 'model'} for m in all_results}
with open(MODELS_DIR / 'ml_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2, default=float)
print(f'Hasil semua model: {MODELS_DIR / "ml_results.json"}')

## 9. Visualisasi Perbandingan Model

In [ ]:
df_all = pd.DataFrame(baseline_results + extra_results)
metrics_to_plot = ['accuracy', 'precision_pos', 'recall_pos', 'f1_pos', 'roc_auc']

fig, ax = plt.subplots(figsize=(11, 5))
df_plot = df_all.set_index('model')[metrics_to_plot]
df_plot.plot(kind='bar', ax=ax, edgecolor='white', width=0.8)
ax.set_title('Perbandingan Performa Model ML', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.set_xlabel('')
ax.set_ylim(0, 1.0)
ax.legend(loc='upper right', fontsize=9)
ax.axhline(0.85, color='gray', linestyle='--', alpha=0.5, label='target accuracy 0.85')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_model_comparison.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, (model, _, needs_scale) in all_models.items():
    X_eval = X_test_scaled if needs_scale else X_test
    proba = model.predict_proba(X_eval)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, proba):.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Semua Model ML', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_roc_curves.png', dpi=150)
plt.show()

In [ ]:
X_best_eval = X_test_scaled if best_needs_scaling else X_test
y_pred_best = best_model.predict(X_best_eval)
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No (0)', 'Yes (1)'], yticklabels=['No (0)', 'Yes (1)'])
ax.set_title(f'Confusion Matrix — {best_name}', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_confusion_matrix_best.png', dpi=150)
plt.show()

## 10. Feature Importance (untuk kemudian dipakai SHAP)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=FEATURE_ORDER).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(9, 6))
    importance.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Top 15 Feature Importance — {best_name}', fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'ml_feature_importance.png', dpi=150)
    plt.show()
elif hasattr(best_model, 'coef_'):
    importance = pd.Series(np.abs(best_model.coef_[0]), index=FEATURE_ORDER).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(9, 6))
    importance.sort_values().plot(kind='barh', ax=ax, color='coral', edgecolor='white')
    ax.set_title(f'Top 15 |Coefficients| — {best_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'ml_feature_importance.png', dpi=150)
    plt.show()

## 11. Ringkasan & Hand-off ke FastAPI

Artifak yang dihasilkan di `ml-api/models/`:

| File | Isi |
|---|---|
| `pulsevera_ml_model.pkl` | Model ML terbaik (joblib) |
| `scaler.pkl` | StandardScaler (hanya dipakai bila best model = LR) |
| `feature_metadata.json` | `FEATURE_ORDER`, nama best model, flag scaling, statistik fitur |
| `ml_results.json` | Ringkasan metrik tiap model |

Lanjut: jalankan `06_deep_learning.ipynb` untuk model DL TensorFlow.